# Tensor Şekil Değiştirme İşlemleri / Manipulating Tensor Shapes

🇹🇷 Derin öğrenmede hataların büyük kısmı yanlış tensor şekillerinden çıkar. Bu defterde bir
tensorün verisini değiştirmeden şeklini değiştirmenin yollarını inceliyoruz: `reshape`, `view`,
`stack`, `squeeze`, `unsqueeze` ve `permute`.

🇬🇧 In deep learning most errors come from mismatched tensor shapes. In this notebook we look at
the ways of changing a tensor's shape without changing its data: `reshape`, `view`, `stack`,
`squeeze`, `unsqueeze` and `permute`.

In [1]:
import torch

## Başlangıç tensorü ve şekli / The starting tensor and its shape

🇹🇷 `torch.arange(1,10,1)` ile 9 elemanlı tek boyutlu bir vektör üretiyoruz. `shape` eleman
sayısını boyut boyut, `ndim` ise boyut sayısını verir.

🇬🇧 With `torch.arange(1,10,1)` we create a one-dimensional vector of 9 elements. `shape` gives the
number of elements per dimension, while `ndim` gives the number of dimensions.

In [2]:
x = torch.arange(1,10,1)

In [3]:
x.shape

torch.Size([9])

In [4]:
x.ndim

1

## reshape ile yeni şekil / Reshaping a tensor

🇹🇷 `reshape` toplam eleman sayısı korunduğu sürece tensorü istediğimiz şekle sokar. 9 elemanlı
vektör `(1,9)` şekline girdiğinde artık iki boyutludur; veri aynı, sadece düzen değişmiştir.

🇬🇧 `reshape` puts the tensor into any shape we want as long as the total number of elements is
preserved. Reshaped to `(1,9)`, the 9-element vector becomes two-dimensional; the data is the
same, only its layout changed.

In [5]:
x_reshaped = x.reshape(1,9)

In [6]:
x_reshaped.shape

torch.Size([1, 9])

In [7]:
x_reshaped.ndim

2

## Görüntü benzeri bir tensor / An image-like tensor

🇹🇷 `torch.randn(224,224,3)` tipik bir görüntü tensorüdür: yükseklik, genişlik ve kanal. Şekli
okumayı alışkanlık hâline getirmek, ilerideki katman hatalarını erkenden yakalamayı sağlar.

🇬🇧 `torch.randn(224,224,3)` is a typical image tensor: height, width and channels. Getting into
the habit of reading shapes helps catch layer errors early on.

In [8]:
y = torch.randn(224,224,3)

In [9]:
y.shape

torch.Size([224, 224, 3])

## reshape kopya mı, görünüm mü? / Does reshape copy or share?

🇹🇷 9 elemanlı vektörü `(3,3)` yapıp bir satırını değiştirdiğimizde orijinal tensorün de
değiştiğini görüyoruz. `reshape` mümkün olduğunda aynı belleği paylaşan bir görünüm döndürür,
yalnızca gerektiğinde kopya çıkarır.

🇬🇧 When we reshape the 9-element vector to `(3,3)` and modify one of its rows, the original tensor
changes too. `reshape` returns a memory-sharing view whenever it can, and only copies when it
has to.

In [10]:
x_reshaped = x.reshape(3,3)

In [11]:
x_reshaped

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

In [12]:
x_reshaped[0] = 20

In [13]:
x_reshaped

tensor([[20, 20, 20],
        [ 4,  5,  6],
        [ 7,  8,  9]])

In [14]:
x

tensor([20, 20, 20,  4,  5,  6,  7,  8,  9])

## view her zaman belleği paylaşır / view always shares memory

🇹🇷 `view` `reshape`'in katı sürümüdür: kopya çıkarmaz, her zaman aynı belleğe bakan bir görünüm
döndürür. Bu yüzden `x_view` üzerinde yapılan değişiklik doğrudan `x`'e de yansır.

🇬🇧 `view` is the strict version of `reshape`: it never copies and always returns a view onto the
same memory. That is why a change made through `x_view` is reflected directly in `x`.

In [15]:
x_view = x.view(9,1)

In [16]:
x_view[0] = 10

In [17]:
x_view

tensor([[10],
        [20],
        [20],
        [ 4],
        [ 5],
        [ 6],
        [ 7],
        [ 8],
        [ 9]])

In [18]:
x

tensor([10, 20, 20,  4,  5,  6,  7,  8,  9])

## Bellekte bitişiklik / Contiguity in memory

🇹🇷 `view` yalnızca bellekte bitişik (contiguous) tensorlerde çalışır. `t()` ile devrik alınan bir
tensor artık bitişik değildir; `is_contiguous()` bunu kontrol eder. Bitişik olmayan bir tensorde
`view` hata verir, bu durumda `reshape` ya da önce `contiguous()` kullanılır.

🇬🇧 `view` only works on tensors that are contiguous in memory. A tensor transposed with `t()` is
no longer contiguous; `is_contiguous()` checks this. Calling `view` on a non-contiguous tensor
raises an error, so we use `reshape`, or call `contiguous()` first.

In [19]:
contiguous_tensor = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])

In [20]:
uncontiguous_tensor = contiguous_tensor.t()

In [21]:
contiguous_tensor = contiguous_tensor.view(1,9)

In [27]:
contiguous_tensor

tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9]])

In [28]:
uncontiguous_tensor = uncontiguous_tensor.view(1,9)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [24]:
uncontiguous_tensor

tensor([[1, 4, 7],
        [2, 5, 8],
        [3, 6, 9]])

In [25]:
uncontiguous_tensor.is_contiguous()

False

In [26]:
contiguous_tensor.is_contiguous()

True

## stack ile tensorleri üst üste koymak / Stacking tensors

🇹🇷 `torch.stack` birden fazla tensoru **yeni bir boyut** açarak birleştirir. `dim` parametresi bu
yeni boyutun nereye ekleneceğini belirler; aynı tensorlerle farklı `dim` değerleri tamamen farklı
şekiller üretir.

🇬🇧 `torch.stack` combines several tensors by creating a **new dimension**. The `dim` parameter
decides where that new dimension is inserted; the same tensors with different `dim` values
produce completely different shapes.

In [34]:
x = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])

In [35]:
torch.stack([x,x,x,x], dim=0)

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]],

        [[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]],

        [[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]],

        [[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])

In [36]:
torch.stack([x,x,x,x], dim=1)

tensor([[[1, 2, 3],
         [1, 2, 3],
         [1, 2, 3],
         [1, 2, 3]],

        [[4, 5, 6],
         [4, 5, 6],
         [4, 5, 6],
         [4, 5, 6]],

        [[7, 8, 9],
         [7, 8, 9],
         [7, 8, 9],
         [7, 8, 9]]])

In [37]:
torch.stack([x,x,x,x], dim=2)

tensor([[[1, 1, 1, 1],
         [2, 2, 2, 2],
         [3, 3, 3, 3]],

        [[4, 4, 4, 4],
         [5, 5, 5, 5],
         [6, 6, 6, 6]],

        [[7, 7, 7, 7],
         [8, 8, 8, 8],
         [9, 9, 9, 9]]])

## squeeze: 1 uzunluklu boyutları atmak / squeeze: dropping size-1 dimensions

🇹🇷 `squeeze()` uzunluğu 1 olan boyutları kaldırır. `(3,3)` şeklindeki bir tensorde böyle bir boyut
olmadığı için şekil değişmez.

🇬🇧 `squeeze()` removes dimensions whose size is 1. A tensor of shape `(3,3)` has no such
dimension, so its shape stays the same.

In [38]:
x

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

In [39]:
x.shape

torch.Size([3, 3])

In [40]:
x.ndim

2

In [41]:
x.squeeze().shape

torch.Size([3, 3])

In [42]:
x.squeeze().ndim

2

## squeeze'in gerçekten iş gördüğü durum / When squeeze actually changes the shape

🇹🇷 `[[1,2,3]]` tensorünün şekli `(1,3)`'tür. Buradaki 1 uzunluklu ilk boyut `squeeze()` ile
kaldırılır ve tensor `(3,)` şekline, yani tek boyutlu bir vektöre döner.

🇬🇧 The tensor `[[1,2,3]]` has shape `(1,3)`. `squeeze()` removes that leading size-1 dimension and
the tensor becomes shape `(3,)`, i.e. a one-dimensional vector.

In [43]:
x = torch.tensor([[1,2,3]])

In [44]:
x.shape

torch.Size([1, 3])

In [45]:
x.ndim

2

In [46]:
x.squeeze()

tensor([1, 2, 3])

In [48]:
x.squeeze().shape

torch.Size([3])

In [49]:
x.squeeze().ndim

1

## unsqueeze: yeni boyut eklemek / unsqueeze: adding a dimension

🇹🇷 `unsqueeze(dim)` belirtilen konuma uzunluğu 1 olan yeni bir boyut ekler; `squeeze`'in tersidir.
Tek bir örneği modele verirken batch boyutu eklemek için sürekli kullanılır.

🇬🇧 `unsqueeze(dim)` adds a new size-1 dimension at the given position; it is the inverse of
`squeeze`. It is used constantly to add a batch dimension when feeding a single sample to a model.

In [50]:
x = torch.tensor([[1,2,3],[4,5,6]])

In [51]:
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [52]:
x.unsqueeze(0)

tensor([[[1, 2, 3],
         [4, 5, 6]]])

In [53]:
x.unsqueeze(0).shape

torch.Size([1, 2, 3])

In [54]:
x.unsqueeze(0).ndim

3

## permute: boyutların sırasını değiştirmek / permute: reordering dimensions

🇹🇷 `permute` boyutları yeniden sıralar. Görüntülerde `(yükseklik, genişlik, kanal)` düzenini
PyTorch'un beklediği `(kanal, yükseklik, genişlik)` düzenine çevirmek için kullanılır. `permute`
de bir görünüm döndürür, veriyi kopyalamaz.

🇬🇧 `permute` reorders dimensions. It is used to convert images from `(height, width, channels)`
into the `(channels, height, width)` layout PyTorch expects. `permute` also returns a view rather
than copying the data.

In [55]:
x = torch.rand(size=(224,224,3))

In [56]:
x.shape

torch.Size([224, 224, 3])

In [57]:
x_permuted = x.permute(2,0,1)

In [58]:
x_permuted.shape

torch.Size([3, 224, 224])

## Tensorlerde indeksleme / Indexing into tensors

🇹🇷 Çok boyutlu bir tensorde indeksleme dıştan içe doğru ilerler: önce en dış boyut, sonra sıradaki.
`:` işareti "bu boyutun tamamı" anlamına gelir, böylece bir satırı ya da sütunu tek seferde
seçebiliriz.

🇬🇧 Indexing a multi-dimensional tensor works from the outside in: the outermost dimension first,
then the next. The `:` symbol means "all of this dimension", letting us select a whole row or
column at once.

In [59]:
a = torch.arange(1,10,1)

In [60]:
a

tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])

In [61]:
a = a.reshape(1,3,3)

In [62]:
a

tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])

In [63]:
a[0][0][0]

tensor(1)

In [64]:
a.shape

torch.Size([1, 3, 3])

In [65]:
a.ndim

3

In [66]:
a[:,0,0]

tensor([1])

In [67]:
a[:,:,0]

tensor([[1, 4, 7]])

## Tekrarlanabilirlik için seed / Seeding for reproducibility

🇹🇷 `torch.manual_seed()` rastgele sayı üretecini sabitler; aynı seed ile her çalıştırmada aynı
tensor üretilir. Deneyleri karşılaştırırken sonucun şansa değil yaptığımız değişikliğe bağlı
olduğunu bilmek için gereklidir.

🇬🇧 `torch.manual_seed()` fixes the random number generator; with the same seed every run produces
the same tensor. This is needed when comparing experiments, so that a difference in results comes
from our change rather than from chance.

In [69]:
RANDOM_SEED = 10
torch.manual_seed(RANDOM_SEED)
random_tensor = torch.rand(3,4)
random_tensor

tensor([[0.4581, 0.4829, 0.3125, 0.6150],
        [0.2139, 0.4118, 0.6938, 0.9693],
        [0.6178, 0.3304, 0.5479, 0.4440]])